# Chapter 06 — Metrics & Evaluation: Live Examples

**Session 2 | Chapter 3 | Duration: ~10 min**

We cover all major metrics visually:
- Regression: MAE, RMSE, R²
- Classification: Confusion matrix, Precision/Recall/F1, ROC/AUC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve
)

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

---
## Part A: Regression Metrics
---

In [ ]:
housing = fetch_california_housing()
X_reg = pd.DataFrame(housing.data, columns=housing.feature_names)
y_reg = housing.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

# Train two regression models
lr = Pipeline([('s', StandardScaler()), ('m', LinearRegression())])
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

lr.fit(X_reg_train, y_reg_train)
rf_reg.fit(X_reg_train, y_reg_train)

y_pred_lr = lr.predict(X_reg_test)
y_pred_rf = rf_reg.predict(X_reg_test)

print('=== Regression Metrics ===')
for name, preds in [('Linear Regression', y_pred_lr), ('Random Forest', y_pred_rf)]:
    mae  = mean_absolute_error(y_reg_test, preds)
    rmse = mean_squared_error(y_reg_test, preds, squared=False)
    r2   = r2_score(y_reg_test, preds)
    print(f'{name:<20}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')

In [ ]:
# Residual plot — visualizing errors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, preds, color) in zip(axes, [
    ('Linear Regression', y_pred_lr, '#3498db'),
    ('Random Forest', y_pred_rf, '#2ecc71')
]):
    residuals = y_reg_test - preds
    ax.scatter(preds, residuals, alpha=0.2, s=8, color=color)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted Value')
    ax.set_ylabel('Residual (Actual − Predicted)')
    ax.set_title(f'{name} — Residual Plot')

plt.suptitle('Residual Plots: Errors should be random around zero', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('If residuals have a pattern → the model is missing something.')

---
## Part B: Classification Metrics
---

In [ ]:
cancer = load_breast_cancer()
X_clf = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_clf = cancer.target

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

clf = Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))])
clf.fit(X_clf_train, y_clf_train)

y_clf_pred  = clf.predict(X_clf_test)
y_clf_proba = clf.predict_proba(X_clf_test)[:, 1]  # probability of class 1 (benign)

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_clf_test, y_clf_pred)
ConfusionMatrixDisplay(cm, display_labels=['malignant', 'benign']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Logistic Regression)', fontsize=12)

# Annotate with labels
tn, fp, fn, tp = cm.ravel()
axes[1].axis('off')
axes[1].text(0.5, 0.7, '📊 From the confusion matrix:', fontsize=13, ha='center', va='center')
axes[1].text(0.5, 0.55, f'True Positives (TP):  {tp}  ← Correctly found cancer', fontsize=11, ha='center', va='center', color='#2ecc71')
axes[1].text(0.5, 0.45, f'True Negatives (TN):  {tn}  ← Correctly found healthy', fontsize=11, ha='center', va='center', color='#2ecc71')
axes[1].text(0.5, 0.35, f'False Positives (FP): {fp}  ← Healthy called malignant', fontsize=11, ha='center', va='center', color='#e67e22')
axes[1].text(0.5, 0.25, f'False Negatives (FN): {fn}  ← Missed malignant tumor ⚠️', fontsize=11, ha='center', va='center', color='#e74c3c')

plt.tight_layout()
plt.show()

In [ ]:
# Precision, Recall, F1
precision = precision_score(y_clf_test, y_clf_pred)
recall    = recall_score(y_clf_test, y_clf_pred)
f1        = f1_score(y_clf_test, y_clf_pred)
accuracy  = (y_clf_pred == y_clf_test).mean()

print(f'Accuracy:  {accuracy:.4f}  →  fraction of correct predictions')
print(f'Precision: {precision:.4f}  →  of predicted benign, how many are actually benign')
print(f'Recall:    {recall:.4f}  →  of all actual benign cases, how many did we catch')
print(f'F1 Score:  {f1:.4f}  →  harmonic mean of precision and recall')
print()
print('Full classification report:')
print(classification_report(y_clf_test, y_clf_pred, target_names=['malignant', 'benign']))

In [ ]:
# ROC Curve + Precision-Recall Curve
fpr, tpr, roc_thresholds = roc_curve(y_clf_test, y_clf_proba)
auc = roc_auc_score(y_clf_test, y_clf_proba)

prec_curve, rec_curve, _ = precision_recall_curve(y_clf_test, y_clf_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'ROC curve (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=13)
axes[0].legend(fontsize=11)

# Precision-Recall Curve
axes[1].plot(rec_curve, prec_curve, color='#e74c3c', linewidth=2.5)
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve', fontsize=13)
axes[1].text(0.05, 0.15, 'Moving left along the curve\n= increasing the threshold\n= fewer positives predicted\n= higher precision, lower recall',
             fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.suptitle('Model Evaluation Curves', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print(f'AUC = {auc:.4f} → excellent discriminative ability!')

In [ ]:
# Model comparison with cross-validation
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models_cv = {
    'Logistic Regression': Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'KNN (k=5)':           Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(5))]),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

print('=== 5-Fold Cross-Validation Comparison ===')
print(f'{"Model":<22}  {"F1 Mean":>10}  {"F1 Std":>8}')
print('-' * 44)

cv_results = {}
for name, m in models_cv.items():
    scores = cross_val_score(m, X_clf, y_clf, cv=5, scoring='f1')
    cv_results[name] = scores
    print(f'{name:<22}  {scores.mean():.4f}     ±{scores.std():.4f}')

# Visualize
fig, ax = plt.subplots(figsize=(9, 4))
means = [cv_results[n].mean() for n in models_cv]
stds  = [cv_results[n].std()  for n in models_cv]
bars = ax.barh(list(models_cv.keys()), means, xerr=stds,
               color='#3498db', alpha=0.75, capsize=5)
ax.set_xlabel('F1 Score (5-Fold CV)', fontsize=12)
ax.set_title('Model Comparison: Cross-Validated F1 Score', fontsize=13)
ax.set_xlim(0.85, 1.0)
plt.tight_layout()
plt.show()